In [ ]:
# job_browsing.ipynb
import tkinter as tk
from tkinter import messagebox, ttk

# Import design elements cleanly from the styles notebook variables
from ipynb.fs.full.styles import (
    BACKGROUND_COLOR, TEXT_COLOR, PRIMARY_COLOR, SECONDARY_COLOR,
    MUTED_TEXT, ERROR_COLOR, SUCCESS_COLOR,
    FONT_TITLE, FONT_SUBTITLE, FONT_BODY, FONT_BUTTON, SKILL_POOL
)

# Database pipeline hooks from your data manager notebook
from ipynb.fs.full.data_manager import get_jobs, save_application, get_applications

class EmployeeDashboard(tk.Frame):
    def __init__(self, master, current_user):
        super().__init__(master, bg=BACKGROUND_COLOR)
        self.master = master
        self.current_user = current_user
        
        # Local component style maps matching main.py implementation
        self.btn_style = {
            "bg": PRIMARY_COLOR, "fg": "#FFFFFF", "font": FONT_BUTTON, "relief": "flat",
            "activebackground": SECONDARY_COLOR, "activeforeground": "#FFFFFF",
            "padx": 12, "pady": 4, "cursor": "hand2"
        }
        
        self.build_ui()

    def build_ui(self):
        # Header Welcome Branding Banner
        header = tk.Label(
            self, 
            text=f"Welcome to your Workspace, {self.current_user['name']}", 
            font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR
        )
        header.pack(pady=15)

        # Tabbed Notebook Layout Frame
        self.tabs = ttk.Notebook(self)
        self.tabs.pack(fill="both", expand=True, padx=15, pady=5)

        # Tab 1: Available Vacancies Explorer Panel
        self.explore_tab = tk.Frame(self.tabs, bg=BACKGROUND_COLOR)
        self.tabs.add(self.explore_tab, text=" Explore Available Roles ")
        self.build_explore_tab()

        # Tab 2: Personal Tracking History Panel
        self.history_tab = tk.Frame(self.tabs, bg=BACKGROUND_COLOR)
        self.tabs.add(self.history_tab, text=" My Submitted Applications ")
        self.build_history_tab()

    def build_explore_tab(self):
        # Scrolling canvas setup
        canvas = tk.Canvas(self.explore_tab, bg=BACKGROUND_COLOR, highlightthickness=0)
        scrollbar = tk.Scrollbar(self.explore_tab, orient="vertical", command=canvas.yview)
        self.scroll_frame = tk.Frame(canvas, bg=BACKGROUND_COLOR)

        self.scroll_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=self.scroll_frame, anchor="nw", width=720)
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        scrollbar.pack(side="right", fill="y")

        self.refresh_job_listings()

    def refresh_job_listings(self):
        # Wipe old rows
        for widget in self.scroll_frame.winfo_children():
            widget.destroy()

        all_jobs = get_jobs()
        my_apps = get_applications(self.current_user["id"])
        applied_job_ids = [str(app["job_id"]) for app in my_apps]

        if not all_jobs:
            tk.Label(
                self.scroll_frame, text="No active vacancy listings found in system database archives.", 
                font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT
            ).pack(pady=50)
            return

        for job in all_jobs:
            j_id = str(job["id"])
            
            box = tk.Frame(self.scroll_frame, bg=BACKGROUND_COLOR, bd=1, relief="solid")
            box.pack(fill="x", padx=10, pady=8, ipady=6)

            tk.Label(box, text=job["title"], font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR).pack(anchor="w", padx=15, pady=2)
            tk.Label(box, text=f"Required Skill Sector Focus: {job['skills']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15)
            tk.Label(box, text=f"Experience Metric Demands: {job['experience']}", font=FONT_BODY, bg=BACKGROUND_COLOR, fg=TEXT_COLOR).pack(anchor="w", padx=15)
            tk.Label(box, text=job["description"], font=FONT_BODY, bg=BACKGROUND_COLOR, fg=MUTED_TEXT, wraplength=580, justify="left").pack(anchor="w", padx=15, pady=4)

            if j_id in applied_job_ids:
                status_lbl = tk.Label(box, text="✓ Application Logged securely", font=FONT_BUTTON, bg=BACKGROUND_COLOR, fg=SUCCESS_COLOR)
                status_lbl.pack(anchor="e", padx=15, pady=5)
            else:
                apply_btn = tk.Button(
                    box, text="Submit Position Application", **self.btn_style,
                    command=lambda target_id=j_id: self.trigger_application(target_id)
                )
                apply_btn.pack(anchor="e", padx=15, pady=5)

    def trigger_application(self, job_id):
        save_application(job_id, self.current_user["id"])
        messagebox.showinfo("Handshake Complete", "Your professional identity metadata has been successfully linked to this job record!")
        self.refresh_job_listings()
        self.refresh_history_tab()

    def build_history_tab(self):
        self.history_container = tk.Frame(self.history_tab, bg=BACKGROUND_COLOR)
        self.history_container.pack(fill="both", expand=True, padx=15, pady=15)
        self.refresh_history_tab()

    def refresh_history_tab(self):
        for widget in self.history_container.winfo_children():
            widget.destroy()

        my_apps = get_applications(self.current_user["id"])

        if not my_apps:
            tk.Label(
                self.history_container, text="You haven't submitted any job applications yet.", 
                font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT
            ).pack(pady=60)
            return

        # Treeview Ledger implementation
        columns = ("id", "title", "experience", "skills")
        table = ttk.Treeview(self.history_container, columns=columns, show="headings", height=15)
        
        table.heading("id", text="Tracking ID")
        table.heading("title", text="Role Title Designation")
        table.heading("experience", text="Experience Profile")
        table.heading("skills", text="Skill Vector Focus")

        table.column("id", width=100, anchor="center")
        table.column("title", width=220, anchor="w")
        table.column("experience", width=150, anchor="center")
        table.column("skills", width=180, anchor="w")

        # Resolve explicit text attributes matching relational rows
        import csv
        import os
        jobs_map = {}
        if os.path.exists("jobs.csv"):
            with open("jobs.csv", "r") as file:
                reader = csv.DictReader(file)
                for row in reader:
                    jobs_map[str(row["id"])] = row

        for index, app in enumerate(my_apps, start=1):
            target_job_id = str(app["job_id"])
            if target_job_id in jobs_map:
                details = jobs_map[target_job_id]
                table.insert("", "end", values=(f"APP-00{index}", details["title"], details["experience"], details["skills"]))
            else:
                table.insert("", "end", values=(f"APP-00{index}", "Unknown Role Listing", "N/A", "N/A"))

        table.pack(fill="both", expand=True)